In [1]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess
subprocess.run(["pip", "install", "unsloth[colab-new]", "unsloth_zoo", "--quiet"])
subprocess.run(["pip", "install", "--no-deps", "trl>=0.12.0", "peft",
                "accelerate", "bitsandbytes", "--quiet"])
print("✓ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.7 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


✓ Done


In [2]:

# ============================================================
# CELL 2 — Login + Load Model + Merge to 16-bit
# ============================================================
from unsloth import FastModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_HOME"]            = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.makedirs("/tmp/hf_cache", exist_ok=True)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

model, tokenizer = FastModel.from_pretrained(
    "dhairyapandya/pathos-gemma4-distilled-rl-histopathology",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Merge LoRA adapter into base model and save as 16-bit
model.save_pretrained_merged(
    "/tmp/merged", tokenizer, save_method="merged_16bit"
)
print("✓ Merged 16-bit model saved to /tmp/merged")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

Splitting model.safetensors (size: 9.54 GB)...


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:35<00:00, 95.54s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [01:11<00:00, 14.38s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/tmp/merged`
✓ Merged 16-bit model saved to /tmp/merged


In [3]:
# ============================================================
# CELL 3 — Free GPU memory (CRITICAL — prevents OOM)
# ============================================================
del model
del tokenizer
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print(f"✓ GPU memory freed. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

✓ GPU memory freed. VRAM used: 0.01 GB


In [4]:
# ============================================================
# CELL 4 — Install llama.cpp + Convert to GGUF Q8_0
# ============================================================
import subprocess, sys, os

# Clone llama.cpp
LLAMA_CPP = "/tmp/llama.cpp"
if not os.path.exists(LLAMA_CPP):
    print("Cloning llama.cpp...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ggerganov/llama.cpp.git", LLAMA_CPP
    ], check=True)
    print("✓ llama.cpp cloned")

# Install ONLY the minimal deps needed for conversion
# (DO NOT install from requirements.txt — it upgrades transformers and breaks Gemma4)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "gguf", "sentencepiece", "protobuf",
    "--quiet", "--no-deps"
], check=True)

# Pin transformers back to working version (5.5.0 supports Gemma4)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "transformers==5.5.0",
    "--quiet"
], check=True)
print("✓ Dependencies installed (transformers pinned to 5.5.0)")

# Convert merged model → Q8_0 GGUF directly
print("Converting to Q8_0 GGUF (10-15 min)...")
result = subprocess.run([
    sys.executable, f"{LLAMA_CPP}/convert_hf_to_gguf.py",
    "/tmp/merged",
    "--outfile", "/tmp/pathos-Q8_0.gguf",
    "--outtype", "q8_0",
], capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("GGUF conversion failed")

size = os.path.getsize("/tmp/pathos-Q8_0.gguf") / 1e9
print(f"\n✓ GGUF ready: /tmp/pathos-Q8_0.gguf ({size:.2f} GB)")

Cloning llama.cpp...


Cloning into '/tmp/llama.cpp'...


✓ llama.cpp cloned
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 2.8 MB/s eta 0:00:00
✓ Dependencies installed (transformers pinned to 5.5.0)
Converting to Q8_0 GGUF (10-15 min)...


✓ GGUF ready: /tmp/pathos-Q8_0.gguf (4.95 GB)


In [5]:
# ============================================================
# CELL 5 — Upload GGUF to HuggingFace
# ============================================================
from huggingface_hub import HfApi, create_repo

GGUF_REPO = "dhairyapandya/pathos-gemma4-distilled-GGUF"

create_repo(GGUF_REPO, token=hf_token, exist_ok=True)
api = HfApi()

print("Uploading GGUF (this may take a few minutes)...")
api.upload_file(
    path_or_fileobj="/tmp/pathos-Q8_0.gguf",
    path_in_repo="pathos-Q8_0.gguf",
    repo_id=GGUF_REPO,
    token=hf_token,
)
print(f"✓ GGUF uploaded to https://huggingface.co/{GGUF_REPO}")

Uploading GGUF (this may take a few minutes)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ GGUF uploaded to https://huggingface.co/dhairyapandya/pathos-gemma4-distilled-GGUF


In [6]:

# ============================================================
# CELL 6 — Generate + Upload Modelfile
# ============================================================
MODELFILE = '''# PathOS Distilled — Ollama Modelfile
# Usage:
#   ollama create pathos -f Modelfile
#   ollama run pathos "Is malignancy present?"

FROM ./pathos-Q8_0.gguf

PARAMETER temperature 0.1
PARAMETER num_predict 300
PARAMETER stop "</answer>"
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.15

SYSTEM """You are PathOS, an expert AI pathologist for H&E stained histopathology analysis.

ABSOLUTE RULES:
1. NEVER hedge. NEVER say "cannot determine", "unable to", "not possible", "uncertain", "not sure", "difficult to". These are BANNED.
2. ALWAYS commit to a specific, definitive answer based on visible features.
3. EVERY response MUST contain exactly one <answer>...</answer> tag.

OUTPUT FORMAT BY QUESTION TYPE:

YES/NO QUESTIONS (starts with is/are/does/do/was/were/has/have/can/could/should):
- First word MUST be "Yes" or "No". No preamble.
- Then: <answer>yes</answer> or <answer>no</answer>
- Then: One sentence of morphological evidence.
- Example: "Yes. <answer>yes</answer> Nuclear pleomorphism with hyperchromatic nuclei visible."

TISSUE IDENTIFICATION (mentions tissue/patch/classify/cell type/stain):
- List 2-3 key morphological features.
- State tissue classification.
- End with <answer>tissue name</answer>
- Example: "Key features: irregular glands, nuclear pleomorphism. <answer>colorectal adenocarcinoma</answer>"

OPEN-ENDED QUESTIONS (all other):
- Observe visible structures briefly.
- State your definitive answer.
- End with <answer>your answer</answer>

REFERENCE EXAMPLES:
Q: "Is nuclear pleomorphism present?"
A: Yes. <answer>yes</answer> Nuclear variation in size and shape with hyperchromatic nuclei.

Q: "Are goblet cells visible?"
A: Yes. <answer>yes</answer> Mucus-secreting columnar cells visible in intestinal mucosa.

Q: "Is carcinoma present?" (normal tissue)
A: No. <answer>no</answer> Regular crypt architecture with intact basement membrane.

Q: "What tissue type is present?" (tumor)
A: Key features: irregular glands, nuclear pleomorphism, hyperchromatic nuclei. <answer>colorectal adenocarcinoma</answer>

Q: "What tissue type is present?" (normal)
A: Key features: regular crypts, columnar epithelium, goblet cells. <answer>normal colon mucosa</answer>

Q: "What is the primary finding?"
A: Dense small round cells with condensed nuclei and scant cytoplasm. <answer>lymphocytic infiltrate</answer>

Q: "What structures are visible?"
A: Univacuolated lipid droplets with peripheral compressed nuclei. <answer>adipose tissue</answer>

Q: "Is necrosis present?"
A: Yes. <answer>yes</answer> Ghost cell outlines and karyolysis indicate tumor necrosis."""

TEMPLATE """{{ if .System }}<start_of_turn>system
{{ .System }}<end_of_turn>
{{ end }}{{ if .Prompt }}<start_of_turn>user
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ end }}{{ .Response }}<end_of_turn>"""
'''

# Save + upload Modelfile
with open("/tmp/Modelfile", "w") as f:
    f.write(MODELFILE)

api.upload_file(
    path_or_fileobj="/tmp/Modelfile",
    path_in_repo="Modelfile",
    repo_id=GGUF_REPO,
    token=hf_token,
)
print("✓ Modelfile uploaded")


✓ Modelfile uploaded


In [7]:

# ============================================================
# CELL 7 — Upload README
# ============================================================
README = """---
language: en
license: apache-2.0
tags:
  - histopathology
  - pathology
  - medical
  - gemma4
  - gguf
  - ollama
base_model: google/gemma-4-e2b-it
---

# PathOS — Distilled GGUF for Ollama

> AI pathologist that runs offline on any laptop via Ollama.

## Quick Start

```bash
# 1. Download pathos-Q8_0.gguf and Modelfile from this repo
# 2. Place both files in the same folder, then:
ollama create pathos -f Modelfile
ollama run pathos "Is malignancy present?"
```

## Model Details

| Property | Value |
|---|---|
| Base model | Gemma 4 E2B (2.6B params) |
| Training | SFT + GRPO distillation |
| Quantization | Q8_0 |
| Source | [dhairyapandya/pathos-gemma4-distilled-rl-histopathology](https://huggingface.co/dhairyapandya/pathos-gemma4-distilled-rl-histopathology) |

## Distilled Techniques

All 6 inference-time techniques baked into the weights:
1. Constrained decoding — always produces `<answer>` tags
2. Anti-hedging — never says "cannot determine"
3. Static RAG — few-shot exemplars in system prompt
4. CoT routing — adapts reasoning to question type
5. Per-type templates — YN/tissue/open formats
6. Format compliance — GRPO reward shaping
"""

api.upload_file(
    path_or_fileobj=README.encode(),
    path_in_repo="README.md",
    repo_id=GGUF_REPO,
    token=hf_token,
)

print(f"✓ README uploaded")
print(f"\n{'='*55}")
print(f"  🎉 DONE!")
print(f"  GGUF repo: https://huggingface.co/{GGUF_REPO}")
print(f"{'='*55}")
print(f"\n  For labs to deploy locally:")
print(f"  1. Download pathos-Q8_0.gguf + Modelfile")
print(f"  2. ollama create pathos -f Modelfile")
print(f"  3. ollama run pathos 'Is malignancy present?'")
print(f"\n  To publish on Ollama registry:")
print(f"  ollama push dhairyapandya/pathos")


✓ README uploaded

  🎉 DONE!
  GGUF repo: https://huggingface.co/dhairyapandya/pathos-gemma4-distilled-GGUF

  For labs to deploy locally:
  1. Download pathos-Q8_0.gguf + Modelfile
  2. ollama create pathos -f Modelfile
  3. ollama run pathos 'Is malignancy present?'

  To publish on Ollama registry:
  ollama push dhairyapandya/pathos
